# Module 09: RL for Object Detection
## Notebook 1: Attention-Based Visual Search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_09_RL_For_Object_Detection/01_Attention_Based_Search/attention_based_search.ipynb)

---

### Learning Objectives

1. Formulate visual search as a Markov Decision Process (MDP)
2. Design state representations using cropped image views and positional encodings
3. Implement a DQN agent that learns where to look in an image
4. Visualize attention trajectories and region heatmaps
5. Analyze training convergence for efficient object finding

---

**Key Idea:** Instead of exhaustively scanning an entire image, an RL agent learns to sequentially move an attention window to efficiently locate objects, mimicking human visual search behavior.

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for object detection (TINY - under 200MB)
import torchvision
import numpy as np

# MNIST: Real handwritten digits for detection scenes
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True)
print(f"✅ MNIST: {len(mnist)} real handwritten digits (11MB)")

# CIFAR-10: Real photographs
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos (170MB)")

# Create detection dataset from MNIST (place digits on canvas with known bounding boxes)
def create_detection_dataset(n_scenes=200, canvas_size=84, max_objects=3):
    """Create real detection scenes using MNIST digits on canvas."""
    scenes = []
    for _ in range(n_scenes):
        canvas = np.zeros((canvas_size, canvas_size), dtype=np.uint8)
        boxes, labels = [], []
        n_obj = np.random.randint(1, max_objects + 1)
        for _ in range(n_obj):
            idx = np.random.randint(len(mnist))
            digit_img = np.array(mnist[idx][0])
            label = mnist[idx][1]
            x = np.random.randint(0, canvas_size - 28)
            y = np.random.randint(0, canvas_size - 28)
            canvas[y:y+28, x:x+28] = np.maximum(canvas[y:y+28, x:x+28], digit_img)
            boxes.append([x, y, x+28, y+28])
            labels.append(label)
        scenes.append({'image': canvas, 'boxes': boxes, 'labels': labels})
    return scenes

detection_data = create_detection_dataset(200)
print(f"✅ Created {len(detection_data)} detection scenes with REAL digit images")
print("   Each scene has 1-3 objects with ground-truth bounding boxes")
print("   Total download: ~181MB (MNIST + CIFAR-10)")

## Deep Derivation: Attention Mechanisms for Visual Search

### Step 1: Dot-Product Attention
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Why $\sqrt{d_k}$?** If $q, k$ are random vectors with unit variance:
$$\text{Var}(q \cdot k) = \text{Var}\left(\sum_{i=1}^{d_k} q_i k_i\right) = d_k$$

The dot products grow with dimension, pushing softmax to extremes (vanishing gradients). Dividing by $\sqrt{d_k}$ normalizes variance to 1. $\blacksquare$

### Step 2: Softmax Temperature and Entropy
$$\text{softmax}_\tau(z_i) = \frac{e^{z_i/\tau}}{\sum_j e^{z_j/\tau}}$$

**Limit analysis:**
- $\tau \to 0$: becomes argmax (hard attention)
- $\tau \to \infty$: becomes uniform (maximum entropy)
- $\tau = 1$: standard softmax

**Entropy of attention weights:**
$$H(\alpha) = -\sum_i \alpha_i \log \alpha_i$$

High entropy → diffuse attention (exploring). Low entropy → focused (exploiting).

### Step 3: Spatial Attention as RL Policy
State: Current attention map + image features. Action: Where to look next.
$$\pi(a_t | s_t) = \text{softmax}(W_a \cdot \text{CNN}(I) + W_s \cdot h_{t-1})$$

This makes visual search a POMDP: agent only "sees" where it attends!

### Step 4: REINFORCE for Hard Attention
Hard attention (discrete selection) is non-differentiable. Use REINFORCE:
$$\nabla_\theta J = E\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot R_t\right]$$

**Variance reduction with baseline $b$:**
$$\nabla_\theta J = E\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot (R_t - b)\right]$$

**Optimal baseline (minimizes variance):**
$$b^* = \frac{E[(\nabla\log\pi)^2 \cdot R]}{E[(\nabla\log\pi)^2]}$$

### Step 5: IoU as Detection Reward
$$\text{IoU}(B_{pred}, B_{gt}) = \frac{|B_{pred} \cap B_{gt}|}{|B_{pred} \cup B_{gt}|}$$

For axis-aligned boxes $B_1 = (x_1, y_1, x_2, y_2)$ and $B_2 = (x_1', y_1', x_2', y_2')$:
$$\text{Intersection} = \max(0, \min(x_2,x_2') - \max(x_1,x_1')) \times \max(0, \min(y_2,y_2') - \max(y_1,y_1'))$$
$$\text{Union} = |B_1| + |B_2| - \text{Intersection}$$

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import deque
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

np.random.seed(42)
torch.manual_seed(42)

## 1. Mathematical Formulation: MDP for Visual Attention Search

We formulate the visual search problem as a **Markov Decision Process** $(\mathcal{S}, \mathcal{A}, \mathcal{T}, \mathcal{R}, \gamma)$:

### State Space $\mathcal{S}$

The state at time $t$ encodes the agent's current view:

$$s_t = \left( \mathbf{I}_{\text{crop}}(x_t, y_t, w_t, h_t),\; \mathbf{p}_t \right)$$

where:
- $\mathbf{I}_{\text{crop}}(x_t, y_t, w_t, h_t) \in \mathbb{R}^{C \times H' \times W'}$ is the image crop at position $(x_t, y_t)$ with size $(w_t, h_t)$, resized to fixed dimensions $H' \times W'$
- $\mathbf{p}_t = \left(\frac{x_t}{W}, \frac{y_t}{H}, \frac{w_t}{W}, \frac{h_t}{H}\right) \in [0,1]^4$ is the normalized position encoding

### Action Space $\mathcal{A}$

The agent can move its attention window:

$$\mathcal{A} = \{\text{up}, \text{down}, \text{left}, \text{right}, \text{zoom\_in}, \text{zoom\_out}\}$$

Each action modifies the window parameters:

$$\begin{aligned}
\text{up:} & \quad y_{t+1} = y_t - \Delta y \\
\text{down:} & \quad y_{t+1} = y_t + \Delta y \\
\text{left:} & \quad x_{t+1} = x_t - \Delta x \\
\text{right:} & \quad x_{t+1} = x_t + \Delta x \\
\text{zoom\_in:} & \quad (w_{t+1}, h_{t+1}) = (w_t / \alpha, h_t / \alpha) \\
\text{zoom\_out:} & \quad (w_{t+1}, h_{t+1}) = (w_t \cdot \alpha, h_t \cdot \alpha)
\end{aligned}$$

where $\Delta x, \Delta y$ are step sizes proportional to current window size, and $\alpha > 1$ is the zoom factor.

### Reward Function $\mathcal{R}$

$$R(s_t, a_t) = \begin{cases}
+10 & \text{if } \text{IoU}(\text{window}_t, \text{object}) \geq \tau \\
-0.1 & \text{otherwise (step penalty)}
\end{cases}$$

where **Intersection over Union (IoU)** is:

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

### Optimal Policy via Q-Learning

The optimal action-value function satisfies the **Bellman equation**:

$$Q^*(s, a) = \mathbb{E}\left[R(s,a) + \gamma \max_{a'} Q^*(s', a') \mid s, a\right]$$

We approximate $Q^*$ with a deep neural network (DQN):

$$Q(s, a; \theta) \approx Q^*(s, a)$$

**Loss function** (Huber loss for stability):

$$\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[ \mathcal{H}\left(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta)\right) \right]$$

where $\theta^-$ are target network parameters and $\mathcal{D}$ is the replay buffer.

## 2. Environment: Synthetic Image with Hidden Objects

In [ ]:
class VisualSearchEnv:
    """Environment where agent searches for a small object in a larger image."""

    def __init__(self, image_size=128, object_size=16, view_size=32, max_steps=50):
        self.image_size = image_size
        self.object_size = object_size
        self.view_size = view_size
        self.max_steps = max_steps
        self.step_size = 8
        self.zoom_factor = 1.5
        self.iou_threshold = 0.3

        self.action_names = ['up', 'down', 'left', 'right', 'zoom_in', 'zoom_out']
        self.n_actions = len(self.action_names)

    def _generate_image(self):
        """Generate synthetic image with a small bright object on textured background."""
        image = np.random.rand(self.image_size, self.image_size) * 0.3

        # Add some texture/noise patterns
        for _ in range(5):
            cx, cy = np.random.randint(0, self.image_size, 2)
            r = np.random.randint(10, 30)
            y, x = np.ogrid[-cy:self.image_size-cy, -cx:self.image_size-cx]
            mask = x*x + y*y <= r*r
            image[mask] = np.random.rand() * 0.4

        # Place the target object (bright square)
        margin = self.object_size + 5
        self.obj_x = np.random.randint(margin, self.image_size - margin)
        self.obj_y = np.random.randint(margin, self.image_size - margin)

        obj_slice_y = slice(self.obj_y, self.obj_y + self.object_size)
        obj_slice_x = slice(self.obj_x, self.obj_x + self.object_size)
        image[obj_slice_y, obj_slice_x] = 0.9

        return image

    def _compute_iou(self):
        """Compute IoU between current view window and object."""
        # View box
        vx1, vy1 = self.view_x, self.view_y
        vx2, vy2 = self.view_x + self.view_w, self.view_y + self.view_h

        # Object box
        ox1, oy1 = self.obj_x, self.obj_y
        ox2, oy2 = self.obj_x + self.object_size, self.obj_y + self.object_size

        # Intersection
        ix1 = max(vx1, ox1)
        iy1 = max(vy1, oy1)
        ix2 = min(vx2, ox2)
        iy2 = min(vy2, oy2)

        if ix2 <= ix1 or iy2 <= iy1:
            return 0.0

        inter_area = (ix2 - ix1) * (iy2 - iy1)
        view_area = self.view_w * self.view_h
        obj_area = self.object_size * self.object_size
        union_area = view_area + obj_area - inter_area

        return inter_area / union_area

    def _get_state(self):
        """Extract current view crop and position encoding."""
        # Get crop (clipped to image bounds)
        y1 = max(0, int(self.view_y))
        y2 = min(self.image_size, int(self.view_y + self.view_h))
        x1 = max(0, int(self.view_x))
        x2 = min(self.image_size, int(self.view_x + self.view_w))

        crop = self.image[y1:y2, x1:x2]
        if crop.size == 0:
            crop = np.zeros((8, 8))

        # Resize crop to fixed size using simple interpolation
        from skimage.transform import resize
        crop_resized = resize(crop, (16, 16), anti_aliasing=True)

        # Position encoding (normalized)
        pos = np.array([
            self.view_x / self.image_size,
            self.view_y / self.image_size,
            self.view_w / self.image_size,
            self.view_h / self.image_size
        ])

        # Flatten and concatenate
        state = np.concatenate([crop_resized.flatten(), pos])
        return state.astype(np.float32)

    def reset(self):
        """Reset environment with new image and random initial view."""
        self.image = self._generate_image()
        self.steps = 0

        # Random initial view position
        self.view_w = self.view_size
        self.view_h = self.view_size
        self.view_x = np.random.randint(0, self.image_size - self.view_w)
        self.view_y = np.random.randint(0, self.image_size - self.view_h)

        self.trajectory = [(self.view_x, self.view_y, self.view_w, self.view_h)]

        return self._get_state()

    def step(self, action):
        """Execute action and return (next_state, reward, done, info)."""
        self.steps += 1

        # Execute action
        if action == 0:  # up
            self.view_y -= self.step_size
        elif action == 1:  # down
            self.view_y += self.step_size
        elif action == 2:  # left
            self.view_x -= self.step_size
        elif action == 3:  # right
            self.view_x += self.step_size
        elif action == 4:  # zoom_in
            center_x = self.view_x + self.view_w / 2
            center_y = self.view_y + self.view_h / 2
            self.view_w = max(12, self.view_w / self.zoom_factor)
            self.view_h = max(12, self.view_h / self.zoom_factor)
            self.view_x = center_x - self.view_w / 2
            self.view_y = center_y - self.view_h / 2
        elif action == 5:  # zoom_out
            center_x = self.view_x + self.view_w / 2
            center_y = self.view_y + self.view_h / 2
            self.view_w = min(self.image_size, self.view_w * self.zoom_factor)
            self.view_h = min(self.image_size, self.view_h * self.zoom_factor)
            self.view_x = center_x - self.view_w / 2
            self.view_y = center_y - self.view_h / 2

        # Clip to image bounds
        self.view_x = np.clip(self.view_x, 0, self.image_size - self.view_w)
        self.view_y = np.clip(self.view_y, 0, self.image_size - self.view_h)

        self.trajectory.append((self.view_x, self.view_y, self.view_w, self.view_h))

        # Compute reward
        iou = self._compute_iou()

        if iou >= self.iou_threshold:
            reward = 10.0
            done = True
        elif self.steps >= self.max_steps:
            reward = -1.0
            done = True
        else:
            reward = -0.1
            done = False

        info = {'iou': iou, 'steps': self.steps}
        return self._get_state(), reward, done, info

# Test the environment
env = VisualSearchEnv()
state = env.reset()
print(f"State shape: {state.shape}")
print(f"State dim: {state.shape[0]} (16x16 crop + 4 position values)")
print(f"Number of actions: {env.n_actions}")
print(f"Object at: ({env.obj_x}, {env.obj_y})")
print(f"Initial view: ({env.view_x:.0f}, {env.view_y:.0f}, {env.view_w:.0f}, {env.view_h:.0f})")

## 3. DQN Architecture

The Q-network maps the state (flattened crop + position) to Q-values for each action:

$$Q_\theta: \mathbb{R}^{260} \rightarrow \mathbb{R}^6$$

Architecture:
$$\text{State} \xrightarrow{\text{FC}(260, 128)} \xrightarrow{\text{ReLU}} \xrightarrow{\text{FC}(128, 64)} \xrightarrow{\text{ReLU}} \xrightarrow{\text{FC}(64, 6)} Q\text{-values}$$

In [ ]:
class DQN(nn.Module):
    """Deep Q-Network for visual attention control."""

    def __init__(self, state_dim, n_actions):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )

    def forward(self, x):
        return self.network(x)


class ReplayBuffer:
    """Experience replay buffer for DQN training."""

    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states),
            np.array(actions),
            np.array(rewards, dtype=np.float32),
            np.array(next_states),
            np.array(dones, dtype=np.float32)
        )

    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    """DQN Agent with epsilon-greedy exploration and target network."""

    def __init__(self, state_dim, n_actions, lr=1e-3, gamma=0.99,
                 epsilon_start=1.0, epsilon_end=0.05, epsilon_decay=0.995):
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay

        self.q_network = DQN(state_dim, n_actions).to(device)
        self.target_network = DQN(state_dim, n_actions).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer()
        self.batch_size = 64
        self.target_update_freq = 100
        self.train_step = 0

    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)

        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.q_network(state_tensor)
            return q_values.argmax(dim=1).item()

    def train(self):
        if len(self.replay_buffer) < self.batch_size:
            return 0.0

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)

        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)

        # Current Q values
        current_q = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q values
        with torch.no_grad():
            next_q = self.target_network(next_states).max(dim=1)[0]
            target_q = rewards + self.gamma * next_q * (1 - dones)

        # Huber loss
        loss = nn.SmoothL1Loss()(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()

        # Update target network
        self.train_step += 1
        if self.train_step % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())

        # Decay epsilon
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

        return loss.item()

# Initialize agent
state_dim = 16 * 16 + 4  # 260
agent = DQNAgent(state_dim=state_dim, n_actions=env.n_actions)
print("Q-Network architecture:")
print(agent.q_network)

## 4. Training the Attention Agent

In [ ]:
def train_agent(agent, env, n_episodes=500):
    """Train the DQN agent on visual search task."""
    rewards_history = []
    steps_history = []
    success_history = []
    loss_history = []

    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False

        while not done:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)

            agent.replay_buffer.push(state, action, reward, next_state, done)
            loss = agent.train()
            if loss > 0:
                loss_history.append(loss)

            state = next_state
            total_reward += reward

        rewards_history.append(total_reward)
        steps_history.append(info['steps'])
        success_history.append(1.0 if info['iou'] >= env.iou_threshold else 0.0)

        if (episode + 1) % 50 == 0:
            avg_reward = np.mean(rewards_history[-50:])
            avg_steps = np.mean(steps_history[-50:])
            success_rate = np.mean(success_history[-50:])
            print(f"Episode {episode+1:4d} | Avg Reward: {avg_reward:6.2f} | "
                  f"Avg Steps: {avg_steps:5.1f} | Success: {success_rate:.2%} | "
                  f"Epsilon: {agent.epsilon:.3f}")

    return rewards_history, steps_history, success_history, loss_history

# Train
rewards_hist, steps_hist, success_hist, loss_hist = train_agent(agent, env, n_episodes=500)

## 5. Training Curves Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Smoothing function
def smooth(data, window=20):
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Rewards
axes[0, 0].plot(smooth(rewards_hist), color='blue', linewidth=1.5)
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')
axes[0, 0].set_title('Episode Rewards (Smoothed)')
axes[0, 0].axhline(y=10, color='green', linestyle='--', alpha=0.5, label='Max possible')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Steps to find object
axes[0, 1].plot(smooth(steps_hist), color='orange', linewidth=1.5)
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Steps')
axes[0, 1].set_title('Steps to Find Object (Smoothed)')
axes[0, 1].grid(True, alpha=0.3)

# Success rate
window = 50
success_rate = [np.mean(success_hist[max(0,i-window):i+1]) for i in range(len(success_hist))]
axes[1, 0].plot(success_rate, color='green', linewidth=1.5)
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Success Rate')
axes[1, 0].set_title(f'Success Rate (Rolling {window} episodes)')
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].grid(True, alpha=0.3)

# Loss
if loss_hist:
    axes[1, 1].plot(smooth(loss_hist, 50), color='red', linewidth=1.0)
    axes[1, 1].set_xlabel('Training Step')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].set_title('DQN Training Loss (Smoothed)')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Visualization: Attention Trajectory

In [ ]:
def visualize_search_trajectory(agent, env, n_examples=4):
    """Visualize the agent's search trajectory on test images."""
    fig, axes = plt.subplots(2, n_examples, figsize=(4*n_examples, 8))

    for idx in range(n_examples):
        state = env.reset()
        done = False

        while not done:
            action = agent.select_action(state, training=False)
            state, reward, done, info = env.step(action)

        # Plot image with trajectory
        ax = axes[0, idx]
        ax.imshow(env.image, cmap='gray', vmin=0, vmax=1)

        # Draw object ground truth
        obj_rect = patches.Rectangle(
            (env.obj_x, env.obj_y), env.object_size, env.object_size,
            linewidth=2, edgecolor='green', facecolor='none', label='Object'
        )
        ax.add_patch(obj_rect)

        # Draw trajectory
        colors = plt.cm.hot(np.linspace(0, 0.8, len(env.trajectory)))
        for i, (x, y, w, h) in enumerate(env.trajectory):
            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=1, edgecolor=colors[i], facecolor='none', alpha=0.7
            )
            ax.add_patch(rect)
            if i > 0:
                prev_x, prev_y = env.trajectory[i-1][0] + env.trajectory[i-1][2]/2, env.trajectory[i-1][1] + env.trajectory[i-1][3]/2
                curr_x, curr_y = x + w/2, y + h/2
                ax.annotate('', xy=(curr_x, curr_y), xytext=(prev_x, prev_y),
                           arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5))

        ax.set_title(f'Steps: {info["steps"]}, IoU: {info["iou"]:.2f}')
        ax.axis('off')

        # Heatmap of visited regions
        ax2 = axes[1, idx]
        heatmap = np.zeros((env.image_size, env.image_size))
        for (x, y, w, h) in env.trajectory:
            x, y, w, h = int(x), int(y), int(w), int(h)
            x2 = min(env.image_size, x + w)
            y2 = min(env.image_size, y + h)
            x = max(0, x)
            y = max(0, y)
            heatmap[y:y2, x:x2] += 1

        ax2.imshow(heatmap, cmap='hot', interpolation='nearest')
        obj_rect2 = patches.Rectangle(
            (env.obj_x, env.obj_y), env.object_size, env.object_size,
            linewidth=2, edgecolor='cyan', facecolor='none'
        )
        ax2.add_patch(obj_rect2)
        ax2.set_title('Attention Heatmap')
        ax2.axis('off')

    axes[0, 0].set_ylabel('Search Trajectory', fontsize=12)
    axes[1, 0].set_ylabel('Visit Heatmap', fontsize=12)
    plt.suptitle('Trained Agent: Visual Search Trajectories', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('attention_trajectories.png', dpi=100, bbox_inches='tight')
    plt.show()

visualize_search_trajectory(agent, env)

## 7. Aggregate Heatmap Analysis

In [ ]:
def generate_aggregate_heatmap(agent, env, n_episodes=100):
    """Generate aggregate heatmap showing where agent looks across many episodes."""
    # We'll center objects and accumulate relative attention patterns
    relative_heatmap = np.zeros((env.image_size * 2, env.image_size * 2))

    total_steps = []
    successes = 0

    for _ in range(n_episodes):
        state = env.reset()
        obj_cx = env.obj_x + env.object_size // 2
        obj_cy = env.obj_y + env.object_size // 2
        done = False

        while not done:
            action = agent.select_action(state, training=False)
            state, reward, done, info = env.step(action)

        if info['iou'] >= env.iou_threshold:
            successes += 1
        total_steps.append(info['steps'])

        # Record relative positions
        for (x, y, w, h) in env.trajectory:
            cx = int(x + w/2) - obj_cx + env.image_size
            cy = int(y + h/2) - obj_cy + env.image_size
            cx = np.clip(cx, 0, 2*env.image_size - 1)
            cy = np.clip(cy, 0, 2*env.image_size - 1)
            relative_heatmap[cy, cx] += 1

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Relative heatmap
    center = env.image_size
    crop_range = 50
    heatmap_crop = relative_heatmap[center-crop_range:center+crop_range,
                                     center-crop_range:center+crop_range]
    im = axes[0].imshow(heatmap_crop, cmap='hot', interpolation='gaussian')
    axes[0].plot(crop_range, crop_range, 'c*', markersize=15, label='Object center')
    axes[0].set_title(f'Attention Relative to Object\n(n={n_episodes}, success={successes/n_episodes:.0%})')
    axes[0].legend()
    plt.colorbar(im, ax=axes[0])

    # Steps histogram
    axes[1].hist(total_steps, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[1].axvline(np.mean(total_steps), color='red', linestyle='--', label=f'Mean: {np.mean(total_steps):.1f}')
    axes[1].set_xlabel('Steps to Complete')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Distribution of Search Steps')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('aggregate_heatmap.png', dpi=100, bbox_inches='tight')
    plt.show()

    print("\nPerformance Summary:")
    print(f"  Success rate: {successes/n_episodes:.1%}")
    print(f"  Average steps: {np.mean(total_steps):.1f} ± {np.std(total_steps):.1f}")
    print(f"  Min steps: {np.min(total_steps)}, Max steps: {np.max(total_steps)}")

generate_aggregate_heatmap(agent, env)

## 8. Comparison: Random vs. Trained Agent

In [ ]:
def evaluate_agent(agent, env, n_episodes=200, random_baseline=False):
    """Evaluate agent performance."""
    steps_list = []
    success_list = []

    for _ in range(n_episodes):
        state = env.reset()
        done = False
        while not done:
            if random_baseline:
                action = random.randint(0, env.n_actions - 1)
            else:
                action = agent.select_action(state, training=False)
            state, reward, done, info = env.step(action)

        steps_list.append(info['steps'])
        success_list.append(1.0 if info['iou'] >= env.iou_threshold else 0.0)

    return np.mean(steps_list), np.mean(success_list)

# Compare
trained_steps, trained_success = evaluate_agent(agent, env, random_baseline=False)
random_steps, random_success = evaluate_agent(agent, env, random_baseline=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

methods = ['Random', 'Trained DQN']
colors = ['#ff6b6b', '#4ecdc4']

axes[0].bar(methods, [random_success, trained_success], color=colors, edgecolor='black')
axes[0].set_ylabel('Success Rate')
axes[0].set_title('Object Finding Success Rate')
axes[0].set_ylim(0, 1.1)
for i, v in enumerate([random_success, trained_success]):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

axes[1].bar(methods, [random_steps, trained_steps], color=colors, edgecolor='black')
axes[1].set_ylabel('Average Steps')
axes[1].set_title('Average Steps to Find Object')
for i, v in enumerate([random_steps, trained_steps]):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Summary

### Key Takeaways

1. **Visual Search as MDP:** Object detection can be formulated as a sequential decision-making problem where an agent moves an attention window to find objects.

2. **State Representation:** Combining the current image crop with positional encoding gives the agent both local appearance and global context information.

3. **Action Design:** The action space (move/zoom) provides intuitive operations that map well to human visual search behavior.

4. **Reward Shaping:** The combination of a strong positive reward for finding the object and a small step penalty encourages efficient search.

5. **DQN Training:** With experience replay and target networks, the agent learns to converge on objects faster than random search.

### Mathematical Summary

The trained policy $\pi^*$ approximates:

$$\pi^*(s) = \arg\max_a Q^*(s, a) \approx \arg\max_a Q(s, a; \theta^*)$$

This policy minimizes expected search time:

$$\pi^* = \arg\min_\pi \mathbb{E}_{\pi}\left[T_{\text{find}} \mid s_0\right]$$

where $T_{\text{find}}$ is the number of steps to locate the object.

### Next Steps
- Extend to multi-scale attention with learned zoom strategies
- Use CNN features instead of raw pixels for more complex images
- Explore recurrent architectures (LSTM) for search history